# humming-kernels

Five examples of the `humming-kernels` package (imported as `humming`): the import and its prebuilt native helpers, the low-bit number formats it understands, a quantized matrix multiply, a comparison of 4-bit and 8-bit weights, and the kernels it compiles at run time. Examples 3 to 5 require a CUDA GPU.

## Example 1 — the import, the version and the prebuilt host-side helpers

In [ ]:
import importlib.metadata, json, os
import humming

print("humming-kernels", humming.__version__)
assert humming.__version__ == importlib.metadata.version("humming-kernels")
native = os.path.join(os.path.dirname(humming.__file__), "_native")
(architecture,) = os.listdir(native)  # one folder, named for this machine's CPU family
with open(os.path.join(native, architecture, "manifest.json")) as handle:
    manifest = json.load(handle)
expected = {"_device_info.abi3.so", "libcubinpatch.so", "libhumming_launcher.so", "nvrtc_compile"}
assert set(manifest) == expected, sorted(manifest)
for helper in expected:
    path = os.path.join(native, architecture, helper)
    assert os.path.isfile(path) and os.path.getsize(path) > 0, f"missing helper: {path}"
with open(os.path.join(native, architecture, "nvrtc_compile"), "rb") as handle:
    assert handle.read(4) == b"\x7fELF"
print(f"OK -- {len(expected)} prebuilt helpers for {architecture}:", sorted(expected))

## Example 2 — the low-bit number formats humming understands (no GPU needed)

In [ ]:
import torch
import humming.dtypes as dtypes

six = dtypes.DataType.from_str("int6")
assert six is not None and six.num_bits == 6 and six.is_integer_type and six.is_signed, vars(six)
assert six.to_str() == "int6" and dtypes.DataType.from_str("uint4").num_bits == 4
fp8 = dtypes.DataType.from_str("float8e4m3")  # 8 bits: 4 of exponent, 3 of mantissa, 1 sign
assert fp8.is_floating_point_type and (fp8.num_bits, fp8.exponent_bits, fp8.mantissa_bits) == (8, 4, 3)
fp4 = dtypes.DataType.from_str("float4e2m1")
assert (fp4.num_bits, fp4.exponent_bits, fp4.mantissa_bits) == (4, 2, 1)
half = dtypes.DataType.from_torch_dtype(torch.float16)
assert half.to_str() == "float16" and dtypes.DataType.from_any(torch.bfloat16).to_str() == "bfloat16"
try:
    dtypes.DataType.from_str("int99x")
except ValueError as error:
    refused = type(error).__name__
else:
    raise AssertionError("an impossible format name was accepted")
print("OK -- int6, uint4, float8e4m3, float4e2m1 and torch's half types parse; 'int99x' raises", refused)

## Example 3 — a quantized matrix multiply on the GPU, close to the exact product

In [ ]:
import os, tempfile, torch
os.environ["HUMMING_CACHE_DIR"] = tempfile.mkdtemp(prefix="humming_nb_")  # before the first kernel is compiled
from humming.layer import HummingLayer
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
torch.manual_seed(0)
weight = torch.randn((256, 512), dtype=torch.float16, device="cuda:0")
inputs = torch.randn((16, 512), dtype=torch.float16, device="cuda:0")
layer = HummingLayer(shape_n=256, shape_k=512, weight_config={"dtype": "int6"}, torch_dtype=torch.float16).cuda()
layer.load_from_unquantized(weight)  # quantize the weight to 6 bits
layer.transform()  # pack it into humming's layout and prepare the kernel
output = layer(inputs)  # adapted from humming's README usage example (0.1.16)
reference = inputs.matmul(weight.T).float()
assert output.shape == (16, 256) and output.dtype == torch.float16, (output.shape, output.dtype)
error = ((output.float() - reference).norm() / reference.norm()).item()
assert error < 0.08, f"the 6-bit product is too far from the exact one: relative error {error:.3f}"
print(f"OK -- 6-bit weights: relative error {error:.3f} against the exact float16 product")

## Example 4 — more bits, less error: 8-bit weights beat 4-bit weights

In [ ]:
import torch
from humming.layer import HummingLayer
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
torch.manual_seed(0)
weight = torch.randn((256, 512), dtype=torch.float16, device="cuda:0")
inputs = torch.randn((16, 512), dtype=torch.float16, device="cuda:0")
reference = inputs.matmul(weight.T).float()
errors = {}
for width in ("int4", "int8"):
    layer = HummingLayer(shape_n=256, shape_k=512, weight_config={"dtype": width}, torch_dtype=torch.float16).cuda()
    layer.load_from_unquantized(weight)
    layer.transform()
    errors[width] = ((layer(inputs).float() - reference).norm() / reference.norm()).item()
assert errors["int8"] < 0.03 < errors["int4"] < 0.3, errors
assert errors["int8"] < errors["int4"] / 4, errors
print("OK -- relative error by weight width:", {k: round(v, 4) for k, v in errors.items()})

## Example 5 — the kernels were compiled on this machine, at run time, and cached

In [ ]:
import glob, json, os, torch
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
cache = os.environ.get("HUMMING_CACHE_DIR", "/nonexistent")
kernels = sorted(glob.glob(os.path.join(cache, "*", "kernel.cubin")))
assert kernels, f"Examples 3 and 4 left no compiled kernel under {cache}"
for path in kernels:
    with open(path, "rb") as handle:
        assert handle.read(4) == b"\x7fELF", path  # GPU machine code in an ELF container
    folder = os.path.dirname(path)
    assert os.path.getsize(os.path.join(folder, "kernel.cu")) > 0  # the generated source it was compiled from
    with open(os.path.join(folder, "cmdline.json")) as handle:
        command = json.dumps(json.load(handle))
    assert "nvcc" not in command, command[:300]  # NVRTC (the run-time compiler library), never NVIDIA's nvcc
print(f"OK -- {len(kernels)} kernels compiled at run time under HUMMING_CACHE_DIR, none by nvcc")